# 03 · RAG Pipeline

**Goal:** Answer questions about Canadian bank filings using the documents indexed in Notebook 2.

This notebook retrieves supporting passages, sends them to an LLM, and saves the answers with citations. We then manually review 15 answers and report accuracy.

In [1]:
%pip install -q "chromadb>=1,<2" sentence-transformers openai python-dotenv pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup

Select `EMBEDDING_PROVIDER = "local"` or `"openai"`. This must match the collection made in Notebook 2. The local option uses a free embedding model to search, but **answer generation still calls the OpenAI API**.

In [2]:
import json
import os
import time
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

ROOT = Path.cwd()
BASE = ROOT.parent if ROOT.name == 'notebooks' else ROOT
load_dotenv(BASE / '.env')

MODELS = BASE / 'models'
REPORTS = BASE / 'reports' / 'rag_pipeline'
REPORTS.mkdir(parents=True, exist_ok=True)

EMBEDDING_PROVIDER = 'local'  # 'local' or 'openai'
CHAT_MODEL = 'gpt-4o-mini'     # change if you prefer another chat model
TOP_K = 5
LOCAL_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'
LOCAL_WINDOW_TOKENS = 220   # must match Notebook 2

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('Add OPENAI_API_KEY to the project .env file before running Step 3.')
if EMBEDDING_PROVIDER not in ('local', 'openai'):
    raise ValueError('EMBEDDING_PROVIDER must be local or openai.')

llm = OpenAI(max_retries=3, timeout=60)
store = chromadb.PersistentClient(path=str(MODELS / 'chroma'))
collection_name = f'bank_filings_{EMBEDDING_PROVIDER}'
try:
    collection = store.get_collection(collection_name, embedding_function=None)
except Exception as exc:
    raise ValueError(f'{collection_name} not found. Run Notebook 2 with this embedding provider.') from exc

if collection.count() == 0:
    raise ValueError('The selected ChromaDB collection is empty. Run Notebook 2 first.')
print(f'Using {collection_name}: {collection.count():,} chunks')
print(f'Answer model: {CHAT_MODEL}')

Using bank_filings_local: 10,196 chunks
Answer model: gpt-4o-mini


## 2. Embed a question and retrieve evidence

Local embeddings split long texts into short windows, then average them. OpenAI uses `text-embedding-3-small`.

In [3]:
def normalize(vectors):
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(lengths, 1e-12)


if EMBEDDING_PROVIDER == 'local':
    local_model = SentenceTransformer(LOCAL_MODEL)


def embed_questions(questions):
    if EMBEDDING_PROVIDER == 'openai':
        response = llm.embeddings.create(model=OPENAI_EMBEDDING_MODEL, input=questions)
        ordered = sorted(response.data, key=lambda item: item.index)
        return normalize(np.array([item.embedding for item in ordered], dtype=np.float32))

    tokenizer = local_model.tokenizer
    window_size = min(LOCAL_WINDOW_TOKENS, local_model.max_seq_length - 2)
    windows, owners, weights = [], [], []
    for index, question in enumerate(questions):
        tokens = tokenizer.encode(question, add_special_tokens=False)
        for start in range(0, len(tokens), window_size):
            piece = tokens[start:start + window_size]
            windows.append(tokenizer.decode(piece))
            owners.append(index)
            weights.append(len(piece))
    if not windows:
        raise ValueError('Questions cannot be empty.')
    window_vectors = local_model.encode(windows, batch_size=32, normalize_embeddings=True)
    vectors = np.zeros((len(questions), window_vectors.shape[1]), dtype=np.float32)
    np.add.at(vectors, owners, window_vectors * np.array(weights)[:, None])
    return normalize(vectors)


def make_filter(**fields):
    conditions = [{key: {'$eq': value}} for key, value in fields.items() if value is not None]
    if not conditions:
        return None
    return conditions[0] if len(conditions) == 1 else {'$and': conditions}


def retrieve(question, filters=None, k=TOP_K):
    vector = embed_questions([question])[0]
    results = collection.query(
        query_embeddings=[vector.tolist()],
        n_results=min(k, collection.count()),
        where=make_filter(**(filters or {})),
        include=['documents', 'metadatas', 'distances'],
    )
    return [
        {
            'chunk_id': chunk_id,
            'text': text,
            'similarity': round(1 - distance, 4),
            **metadata,
        }
        for chunk_id, text, distance, metadata in zip(
            results['ids'][0], results['documents'][0],
            results['distances'][0], results['metadatas'][0]
        )
    ]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 3. Build the prompt and generate an answer

Citations use numbered source labels like `[S1]` so they can be checked against the retrieved text. When evidence is missing, the answer should say so.

In [4]:
SYSTEM_PROMPT = """You are a Canadian bank compliance research assistant.
Answer only from the supplied bank filing excerpts. Do not guess or add outside facts.
If the excerpts do not answer the question, say: "The provided filings do not contain enough information to answer this question."
Cite every factual claim using source labels such as [S1] or [S2].
Use only labels that appear in the supplied context.
Be careful about the bank, filing year, and financial figures.
Keep your answer clear and concise. This is a first-pass research tool, not a final compliance decision."""


def source_name(hit):
    quarter = hit.get('quarter')
    period = f' Q{quarter}' if quarter not in (None, '', 0, '0') else ''
    return (f"{hit.get('bank', hit.get('bank_code', 'Unknown bank'))} | "
            f"{hit.get('document_type', 'filing')} | "
            f"{hit.get('fiscal_year', 'unknown year')}{period} | "
            f"{hit.get('section') or 'Unspecified section'} | "
            f"pages {hit.get('page_start', '?')}-{hit.get('page_end', '?')}")


def build_context(hits):
    return '\n\n'.join(
        f"[S{i}] {source_name(hit)}\nChunk ID: {hit['chunk_id']}\n{hit['text']}"
        for i, hit in enumerate(hits, 1)
    )


def answer_question(question, filters=None):
    started = time.time()
    hits = retrieve(question, filters)
    if not hits:
        return {
            'question': question,
            'answer': 'No filing passages matched the requested filters.',
            'sources': [], 'seconds': round(time.time() - started, 2),
            'input_tokens': 0, 'output_tokens': 0,
        }

    prompt = f"FILING EXCERPTS:\n{build_context(hits)}\n\nQUESTION:\n{question}"
    response = llm.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
    )
    answer = response.choices[0].message.content or ''
    if response.choices[0].finish_reason == 'length':
        print('Warning: model response was truncated.')
    return {
        'question': question,
        'answer': answer,
        'sources': [
            {'label': f'S{i}', **hit} for i, hit in enumerate(hits, 1)
        ],
        'seconds': round(time.time() - started, 2),
        'input_tokens': response.usage.prompt_tokens if response.usage else None,
        'output_tokens': response.usage.completion_tokens if response.usage else None,
    }

## 4. Try one question first

Check whether the answer cites real passages before running the full test.

In [5]:
sample = answer_question(
    'What credit risks did RBC discuss in its annual report?',
    {'bank_code': 'RBC', 'document_type': 'annual_report'},
)
print(sample['answer'])
print('\nRetrieved sources:')
for source in sample['sources']:
    print(f"[{source['label']}] {source_name(source)} (similarity: {source['similarity']})")

RBC discussed several aspects of credit risk in its annual report, including:

1. **Definition of Credit Risk**: Credit risk is defined as the risk of loss associated with an obligor’s potential inability or unwillingness to fulfill its contractual obligations, which may arise from primary obligors (e.g., issuers, debtors) or secondary obligors (e.g., guarantors) and includes counterparty credit risk from both trading and non-trading activities [S1][S2].

2. **Management Structure**: The responsibility for managing credit risk is shared across the organization, following a three lines of defence governance model. The Board approves the credit risk appetite, and credit transactions exceeding certain limits require approval from the Risk Committee of the Board [S1][S2].

3. **Objectives for Managing Credit Risk**: RBC aims to ensure credit quality is not compromised for growth, manage credit risks in transactions and portfolios, avoid excessive concentrations in correlated credit risks, 

## 5. Test 15 compliance questions

Tests cover five banks and five risk categories. They use bank filters to avoid mixing evidence from different institutions.

In [6]:
questions = [
    ('RBC', 'credit', 'What credit risks and loan-loss provisions does RBC discuss?'),
    ('RBC', 'market', 'How does RBC describe its market risk exposure?'),
    ('RBC', 'climate', 'What climate-related risks does RBC identify?'),
    ('TD', 'operational', 'What cybersecurity and operational risks does TD discuss?'),
    ('TD', 'credit', 'What risks does TD identify in its residential mortgage lending?'),
    ('TD', 'liquidity', 'How does TD discuss liquidity and funding risk?'),
    ('BMO', 'liquidity', 'What liquidity risk management measures does BMO describe?'),
    ('BMO', 'market', 'What does BMO say about interest-rate risk?'),
    ('BMO', 'operational', 'What operational risks does BMO disclose?'),
    ('BNS', 'climate', 'What physical and transition climate risks does Scotiabank discuss?'),
    ('BNS', 'credit', 'What credit risks does Scotiabank report?'),
    ('BNS', 'market', 'What market risks does Scotiabank describe?'),
    ('CIBC', 'credit', 'What housing and mortgage credit risks does CIBC identify?'),
    ('CIBC', 'operational', 'What cyber and technology-related risks does CIBC disclose?'),
    ('CIBC', 'liquidity', 'How does CIBC describe its liquidity risk management?'),
]

BANK_NAMES = {'RBC': 'RBC', 'TD': 'TD', 'BMO': 'BMO', 'BNS': 'Scotiabank', 'CIBC': 'CIBC'}
tests = pd.DataFrame([
    {'id': f'q{i:02d}', 'bank_code': bank, 'bank': BANK_NAMES[bank],
     'risk': risk, 'question': question}
    for i, (bank, risk, question) in enumerate(questions, start=1)
])
display(tests)

,id,bank_code,bank,risk,question
0,q01,RBC,RBC,credit,What credit risks and loan-loss provisions doe...
1,q02,RBC,RBC,market,How does RBC describe its market risk exposure?
2,q03,RBC,RBC,climate,What climate-related risks does RBC identify?
3,q04,TD,TD,operational,What cybersecurity and operational risks does ...
4,q05,TD,TD,credit,What risks does TD identify in its residential...
5,q06,TD,TD,liquidity,How does TD discuss liquidity and funding risk?
6,q07,BMO,BMO,liquidity,What liquidity risk management measures does B...
7,q08,BMO,BMO,market,What does BMO say about interest-rate risk?
8,q09,BMO,BMO,operational,What operational risks does BMO disclose?
9,q10,BNS,Scotiabank,climate,What physical and transition climate risks doe...


## 6. Run the test and save responses

This cell makes 15 chat API calls, so it incurs costs. If you have already saved the results, it loads them instead. Set `RERUN_TESTS = True` if you want to generate fresh results.

In [7]:
RESULTS_FILE = REPORTS / f'answers_{EMBEDDING_PROVIDER}.json'
RERUN_TESTS = False

if RESULTS_FILE.exists() and not RERUN_TESTS:
    test_results = json.loads(RESULTS_FILE.read_text(encoding='utf-8'))
    print('Loaded saved test answers')
else:
    test_results = []
    for test in tqdm(tests.to_dict('records'), desc='Answering questions'):
        result = answer_question(test['question'], {'bank_code': test['bank_code']})
        test_results.append({**test, **result})
        # Save after each question in case the notebook is interrupted.
        RESULTS_FILE.write_text(json.dumps(test_results, indent=2, ensure_ascii=False), encoding='utf-8')

summary = pd.DataFrame([
    {'id': result['id'], 'bank': result['bank'], 'risk': result['risk'],
     'question': result['question'], 'answer': result['answer'],
     'source_count': len(result['sources']), 'seconds': result['seconds']}
    for result in test_results
])
summary.to_csv(REPORTS / f'answers_{EMBEDDING_PROVIDER}.csv', index=False)
display(summary)

Loaded saved test answers


,id,bank,risk,question,answer,source_count,seconds
0,q01,RBC,credit,What credit risks and loan-loss provisions doe...,The provided filings discuss several aspects o...,5,4.19
1,q02,RBC,market,How does RBC describe its market risk exposure?,RBC describes its market risk exposure through...,5,3.59
2,q03,RBC,climate,What climate-related risks does RBC identify?,RBC identifies two main categories of climate-...,5,2.78
3,q04,TD,operational,What cybersecurity and operational risks does ...,Toronto-Dominion Bank discusses several cybers...,5,4.25
4,q05,TD,credit,What risks does TD identify in its residential...,TD identifies several risks in its residential...,5,3.04
5,q06,TD,liquidity,How does TD discuss liquidity and funding risk?,TD discusses liquidity and funding risk throug...,5,3.01
6,q07,BMO,liquidity,What liquidity risk management measures does B...,BMO describes several liquidity risk managemen...,5,4.42
7,q08,BMO,market,What does BMO say about interest-rate risk?,BMO describes structural interest rate risk as...,5,2.18
8,q09,BMO,operational,What operational risks does BMO disclose?,"BMO discloses several operational risks, inclu...",5,3.49
9,q10,Scotiabank,climate,What physical and transition climate risks doe...,Scotiabank discusses several physical and tran...,5,3.56


## 7. Inspect answers and retrieved evidence

Run this cell with any question ID. Look at the actual supporting text, not just the similarity score.

In [8]:
QUESTION_ID = 'q01'
matching = next((item for item in test_results if item['id'] == QUESTION_ID), None)
if matching is None:
    print('Question not found. Run the test cell first.')
else:
    print('QUESTION:', matching['question'])
    print('\nANSWER:', matching['answer'])
    for source in matching['sources']:
        print(f"\n[{source['label']}] {source_name(source)}")
        print('Chunk ID:', source['chunk_id'])
        print('Text:', source['text'][:1800])

QUESTION: What credit risks and loan-loss provisions does RBC discuss?

ANSWER: The provided filings discuss several aspects of credit risks and loan-loss provisions at RBC:

1. **Credit Risk Management**: RBC establishes risk limits at various levels, including single name, regional, country, and industrial sector limits, to manage credit exposures and ensure alignment with their risk appetite. They actively manage these limits to avoid undue concentration risk, which arises from large exposures that could be similarly affected by economic or political changes [S1].

2. **Concentration Risk**: This is defined as the risk from overexposure in particular industry sectors, countries, or credit products, reflecting the potential for credit deterioration and default to be highly correlated. Credit concentration limits are regularly reviewed considering the business, economic, financial, and regulatory environments [S4].

3. **Loan Forbearance**: RBC has specialized groups and formalized po

## 8. Manual review

For each question, open the answer and source excerpts above. Then fill in the CSV file:

- `answerable`: `1` if the filings contain enough evidence, `0` if not.
- `correct`: `1` if the answer is supported by the evidence (or correctly says it cannot answer), otherwise `0`.
- `citations_accurate`: `1` if every factual claim cites the right supporting excerpt, otherwise `0`.
- `failure_type`: use `retrieval`, `generation`, `missing_data`, or leave blank.

**Important:** `correct` and `citations_accurate` are manual judgments. The notebook does not automatically claim the model is accurate.

In [9]:
REVIEW_FILE = REPORTS / f'manual_review_{EMBEDDING_PROVIDER}.csv'
review_rows = summary[['id', 'bank', 'risk', 'question', 'answer']].copy()

if REVIEW_FILE.exists():
    old = pd.read_csv(REVIEW_FILE, dtype=str, keep_default_na=False)
    for column in ['answerable', 'correct', 'citations_accurate', 'failure_type', 'notes']:
        if column not in old:
            old[column] = ''
    review_rows = review_rows.merge(
        old[['id', 'answerable', 'correct', 'citations_accurate', 'failure_type', 'notes']].drop_duplicates('id'),
        on='id', how='left'
    ).fillna('')
else:
    for column in ['answerable', 'correct', 'citations_accurate', 'failure_type', 'notes']:
        review_rows[column] = ''

review_rows.to_csv(REVIEW_FILE, index=False)
print('Review file:', REVIEW_FILE)
display(review_rows[['id', 'bank', 'risk', 'answerable', 'correct', 'citations_accurate', 'failure_type']])

Review file: c:\Users\ashym\Documents\GitHub\jarvis_data_eng_YashMehrotra\ml_3\reports\rag_pipeline\manual_review_local.csv


,id,bank,risk,answerable,correct,citations_accurate,failure_type
0,q01,RBC,credit,1,1,,
1,q02,RBC,market,0,0,,
2,q03,RBC,climate,1,1,,
3,q04,TD,operational,1,1,,
4,q05,TD,credit,1,1,,
5,q06,TD,liquidity,1,1,,
6,q07,BMO,liquidity,1,1,,
7,q08,BMO,market,1,0,,
8,q09,BMO,operational,1,1,,
9,q10,Scotiabank,climate,1,1,,


## 9. Accuracy and failure analysis

After filling in the review CSV, rerun this cell. Blank scores are not counted as correct or incorrect. Report both overall answer correctness and citation accuracy, and investigate individual failures.

In [10]:
review = pd.read_csv(REVIEW_FILE, dtype=str, keep_default_na=False)
for column in ['answerable', 'correct', 'citations_accurate']:
    values = set(review[column].dropna()) - {''}
    if not values.issubset({'0', '1'}):
        raise ValueError(f'{column} must contain only 0, 1 or blank.')
    review[column] = pd.to_numeric(review[column], errors='coerce')

if review['correct'].notna().sum() == 0:
    print('No questions reviewed yet. Fill in the CSV and rerun this cell.')
else:
    print(f"Answer accuracy: {review['correct'].mean():.1%} "
          f"({review['correct'].notna().sum()} reviewed)")
    if review['citations_accurate'].notna().sum():
        print(f"Citation accuracy: {review['citations_accurate'].mean():.1%} "
              f"({review['citations_accurate'].notna().sum()} reviewed)")
    print('\nAccuracy by risk:')
    display(review.groupby('risk')[['correct', 'citations_accurate']].mean())
    print('\nQuestions needing attention:')
    display(review[(review['correct'] == 0) | (review['citations_accurate'] == 0)]
            [['id', 'bank', 'risk', 'failure_type', 'notes']])
    print('\nFailure types:')
    print(review.loc[review['failure_type'].str.strip() != '', 'failure_type'].value_counts())

review.to_csv(REPORTS / f'review_scores_{EMBEDDING_PROVIDER}.csv', index=False)

Answer accuracy: 80.0% (15 reviewed)

Accuracy by risk:


,correct,citations_accurate
risk,,
climate,1.000000,NaN
credit,1.000000,NaN
liquidity,0.666667,NaN
market,0.333333,NaN
operational,1.000000,NaN



Questions needing attention:


,id,bank,risk,failure_type,notes
1,q02,RBC,market,,
7,q08,BMO,market,,
14,q15,CIBC,liquidity,,



Failure types:
Series([], Name: count, dtype: int64)
